#### Fabric Demo — Carbon Intensity Seed Notebook

##### Generates synthetic carbon intensity records and writes them as a Delta table directly into ManufacturingLakehouse. 
Run ONCE before the demo. No loop, no Event Hub, no API, no credentials. The notebook uses the Lakehouse that is attached to it — Spark handles everything automatically when the notebook is opened from the Lakehouse.

- **What gets created**:
ManufacturingLakehouse → Tables → `carbon_intensity`  (Delta table)

- **Why Lakehouse and not KQL**:
The data agent can query BOTH the KQL database (real-time energy readings) and this Lakehouse Delta table in the same conversation. That demonstrates Fabric's unified analytics story — one agent, two data stores, one answer.

In [1]:
from datetime import datetime, timezone, timedelta
import random

history_mins = 360   # 6 hours of history — enough for trend questions

# Normal grid state
normal_gco2 = 350   # gCO2/kWh — typical mixed grid (coal + gas + some renewables)
normal_fossil_pct = 52
normal_renewable_pct = 38

# Renewable surge state — wind/solar peak 45–60 minutes ago
# Placed in the past so the agent can compare "during the surge" vs "now"
surge_gco2 = 220   # gCO2/kWh — renewables dominating
surge_fossil_pct = 27
surge_renewable_pct = 66
surge_start_mins_ago = 60    # surge started 60 minutes ago
surge_end_mins_ago = 45    # surge ended 45 minutes ago (15 min window)

def add_noise(value, pct=3):
    """±3% random variation so values look natural, not flat."""
    return round(value * (1 + random.uniform(-pct / 100, pct / 100)), 1)

now = datetime.now(timezone.utc)
records = []

for i in range(history_mins, 0, -1):
    ts       = now - timedelta(minutes=i)
    in_surge = surge_end_mins_ago <= i <= surge_start_mins_ago

    records.append((
        ts,
        "GR",
        add_noise(surge_gco2 if in_surge else normal_gco2),
        add_noise(surge_fossil_pct if in_surge else normal_fossil_pct),
        add_noise(surge_renewable_pct if in_surge else normal_renewable_pct),
    ))

print(f"Generated {len(records)} records.")
print(f"From: {records[0][0].strftime('%H:%M UTC')}")
print(f"To: {records[-1][0].strftime('%H:%M UTC')}")
print(f"Normal gCO2: ~{normal_gco2} gCO2/kWh")
print(f"Surge gCO2: ~{surge_gco2} gCO2/kWh  "f"({surge_start_mins_ago}–{surge_end_mins_ago} min ago)")


StatementMeta(, 903bffd3-3e0c-421c-8f7e-998184d3f576, 3, Finished, Available, Finished, False)

Generated 360 records.
  From        : 03:13 UTC
  To          : 09:12 UTC
  Normal gCO2 : ~350 gCO2/kWh
  Surge gCO2  : ~220 gCO2/kWh  (60–45 min ago)


In [2]:
# =============================================================================
# Write to Lakehouse as a Delta table
#
# mode("overwrite") means re-running this cell is safe — it replaces the data.
# =============================================================================
from pyspark.sql.types import StructType, StructField, TimestampType, StringType, DoubleType

schema = StructType([
    StructField("Timestamp", TimestampType(), False),
    StructField("GridZone", StringType(),    False),
    StructField("GCo2PerKwh", DoubleType(),    False),
    StructField("FossilFuelPct", DoubleType(),    False),
    StructField("RenewablePct", DoubleType(),    False),
])
variables = notebookutils.variableLibrary.getLibrary("var_library_rti")

df = spark.createDataFrame(records, schema)

df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(f"{variables.lakehouse_abfss}/dbo/carbon_intensity")

print(f"Written {df.count()} rows to ManufacturingLakehouse → Tables → carbon_intensity")

StatementMeta(, 903bffd3-3e0c-421c-8f7e-998184d3f576, 4, Finished, Available, Finished, False)

Written 360 rows to ManufacturingLakehouse → Tables → carbon_intensity


In [3]:
# =============================================================================
# Quick verification
# =============================================================================
result = spark.read.format("delta").load(f"{variables.lakehouse_abfss}/dbo/carbon_intensity")

display(result)
print("Table is ready. Refresh the Lakehouse explorer to see carbon_intensity.")

StatementMeta(, 903bffd3-3e0c-421c-8f7e-998184d3f576, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e2f05523-58b6-4792-a8da-0b29c6c4cfdc)

Table is ready. Refresh the Lakehouse explorer to see carbon_intensity.
